In [30]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import stylia
from stylia import CategoricalPalette

stylia.set_format("slide")
stylia.set_style("ersilia")
pal = CategoricalPalette("ersilia")
plt.rcParams["figure.dpi"] = 300

REPO_ROOT    = os.path.abspath(os.path.join(os.getcwd(), ".."))
pathogens    = pd.read_csv(os.path.join(REPO_ROOT, "config", "pathogens.csv"))
code_to_name = dict(zip(pathogens["code"], pathogens["pathogen"]))
reports      = pd.read_csv(os.path.join(REPO_ROOT, "output", "results", "10_reports.csv"))

In [ ]:
pathogen = "ecoli"

# ── load data ──
src12        = os.path.join(REPO_ROOT, "output", "results", "12_drugbank",     f"{pathogen}.csv")
src13        = os.path.join(REPO_ROOT, "output", "results", "13_consensus",    f"{pathogen}.csv")
src14_models = os.path.join(REPO_ROOT, "output", "results", "14_correlations", f"{pathogen}_models.csv")
src14_cons   = os.path.join(REPO_ROOT, "output", "results", "14_correlations", f"{pathogen}_consensus.csv")
src14_recap  = os.path.join(REPO_ROOT, "output", "results", "14_recapitulation", f"{pathogen}.csv")

for src in (src12, src13, src14_models, src14_cons):
    if not os.path.isfile(src) or os.path.getsize(src) == 0:
        raise FileNotFoundError(src)

df12     = pd.read_csv(src12)
df13     = pd.read_csv(src13)
df_pairs = pd.read_csv(src14_models)
df_cons  = pd.read_csv(src14_cons)
df_recap = pd.read_csv(src14_recap)
df_recap_off = df_recap[df_recap["model_scorer"] != df_recap["model_binarized"]]

model_cols = [c for c in df12.columns if c != "smiles"]
report_p   = reports[reports["pathogen"] == pathogen].set_index("model_name")
N          = len(model_cols)
rng        = np.random.default_rng(42)
width      = max(0.5, N * 0.06)
width      = 0.3
w          = min(0.35, max(0.15, 1.0 / N))

def _plot_col(ax, values, pos, bw, color):
    jitter = [pos + rng.uniform(-bw, bw) for _ in values]
    ax.scatter(jitter, values, color=color, s=4, alpha=0.6, lw=0)
    stats = dict(
        med=np.median(values), q1=np.percentile(values, 25),
        q3=np.percentile(values, 75), whislo=np.percentile(values, 5),
        whishi=np.percentile(values, 95), fliers=[],
    )
    bp = ax.bxp([stats], positions=[pos], widths=bw * 2, patch_artist=True, showfliers=False)
    bp["boxes"][0].set_facecolor("none")
    bp["boxes"][0].set_linewidth(0.4)
    for elem in ["whiskers", "caps", "medians"]:
        for line in bp[elem]:
            line.set_color("k")
            line.set_linewidth(0 if elem == "caps" else 0.4)

fig, axs = stylia.create_figure(4, 1, width=width, height=0.6)
fig.suptitle(f"{code_to_name[pathogen]} models ({N}) vs. DrugBank compounds ({len(df12)} compounds)", fontsize=11, y=1)

# ── [0, 0] DrugBank prob_rank scores ──
ax = axs.next()
ax.set_ylabel("prob rank")
ax.set_ylim([0, 1])
ax.set_xlim([-0.7, N - 0.3])
for i, model in enumerate(model_cols):
    _plot_col(ax, df12[model].dropna().values, i, w, pal.get(8)[4])
    if model in report_p.index:
        c = report_p.loc[model, "decision_cutoff_rank"]
        ax.plot([i - w * 2, i + w * 2], [c, c], lw=0.8, c="k", linestyle="dotted")
ax.set_xticks(range(N))
ax.set_xticklabels(model_cols, rotation=90, size=6)
ax.set_xlabel(None)

# ── [1, 0] Consensus scores ──
ax = axs.next()
excl_cols = [c for c in df13.columns if c.startswith("excluded_")]
all_cols  = excl_cols + ["consensus_score"]
xlabels   = [c.replace("excluded_", "") for c in excl_cols] + ["global"]
NC  = len(all_cols)
w_c = min(0.35, max(0.15, 1.0 / NC))
ax.set_ylabel("consensus score")
ax.set_ylim([0.2, 0.8])
ax.set_xlim([-0.7, NC - 0.3])
for i, col in enumerate(all_cols):
    color = pal.get(2)[1] if col == "consensus_score" else pal.get(8)[4]
    _plot_col(ax, df13[col].dropna().values, i, w_c, color)
ax.set_xticks(range(NC))
ax.set_xticklabels(xlabels, rotation=90, size=6)
ax.set_xlabel(None)

# ── [1, 1] Model vs consensus Spearman ──
ax = axs.next()
ax.set_ylabel("Spearman r")
ax.set_ylim([0, 1])
ax.set_xlim([-0.7, N - 0.3])
for i, row in df_cons.iterrows():
    ax.bar(i, row["spearman_r_global"],   width=w * 3, color=pal.get(8)[3], ec="k", lw=0.7)
    ax.bar(i, row["spearman_r_excluded"], width=w * 3, color=pal.get(2)[1], ec="k", lw=0.7)
ax.set_xticks(range(N))
ax.set_xticklabels(df_cons["model"].tolist(), rotation=90, size=6)
ax.set_xlabel(None)
ax.legend(handles=[
    mpatches.Patch(color=pal.get(2)[1], label="vs excl. consensus"),
    mpatches.Patch(color=pal.get(8)[3], label="vs global consensus"),
], fontsize=8)


# ── [3] AUROC recapitulation — 4 thresholds ──
THRESH_COLS   = ["auroc_1pct", "auroc_5pct", "auroc_10pct", "auroc_25pct"]
THRESH_LABELS = ["1%", "5%", "10%", "25%"]
df_recap_off  = df_recap[df_recap["model_scorer"] != df_recap["model_binarized"]]
colors        = pal.get(4)

ax = axs.next()
ax.set_xlabel("AUROC")
ax.set_ylabel("Count")
ax.set_xlim([0.2, 1])
bins = np.arange(0, 1.1, 0.02)
for col, label, color in zip(THRESH_COLS, THRESH_LABELS, colors):
    ax.hist(df_recap_off[col].dropna().values, bins=bins, alpha=0.6, label=label, color=color)
ax.axvline(0.5, lw=0.6, ls="--", color="k", alpha=0.4)
ax.legend(title="Threshold", fontsize=8, ncol=2)


plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/aloy/home/acomajuncosa/Ersilia/chembl-antimicrobial-models/output/results/14_recapitulation/ecoli.csv'